In [1]:
import os
from datetime import datetime, timezone
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
DB_URL = os.getenv("LOCAL_DATABASE_URL")
if DB_URL and DB_URL.startswith("postgresql://"):
    DB_URL = DB_URL.replace("postgresql://", "postgresql+psycopg2://", 1)

engine = create_engine(DB_URL)
pipeline_run_id = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

print("Database connection berhasil")
print("Pipeline run ID:", pipeline_run_id)


Database connection berhasil
Pipeline run ID: 20260924073702


## 1. Build `product_daily`

In [4]:
order_360 = pd.read_sql(
    "SELECT * FROM gold.order_360",
    engine
)

order_360.to_sql(
    "_tmp_order_360",
    engine,
    if_exists="replace",
    index=False
)

product_daily = pd.read_sql("""
WITH sales AS (
    SELECT
        oi.product_id,
        o.order_date::date AS metric_date,
        SUM(oi.quantity) AS units_sold,
        COUNT(DISTINCT oi.order_id) AS order_count,
        SUM(oi.quantity * oi.unit_price) AS gross_revenue
    FROM _tmp_order_360 o
    JOIN silver.order_items oi
        ON oi.order_id = o.order_id
    GROUP BY
        oi.product_id,
        o.order_date::date
),

refund_orders AS (
    SELECT DISTINCT
        order_id
    FROM _tmp_order_360
    WHERE refunded_amount > 0
),

refund_units AS (
    SELECT
        oi.product_id,
        o.order_date::date AS metric_date,
        SUM(oi.quantity) AS refunded_units
    FROM silver.order_items oi
    JOIN _tmp_order_360 o
        ON o.order_id = oi.order_id
    JOIN refund_orders r
        ON r.order_id = oi.order_id
    GROUP BY
        oi.product_id,
        o.order_date::date
),

product_dates AS (
    SELECT
        product_id,
        snapshot_date::date AS metric_date,
        SUM(available_quantity) AS available_inventory,
        SUM(reserved_quantity) AS reserved_inventory
    FROM silver.inventory_snapshots
    GROUP BY
        product_id,
        snapshot_date::date
),

cats AS (
    SELECT
        product_id,
        category_name,
        valid_from_utc,
        valid_to_utc,
        ROW_NUMBER() OVER (
            PARTITION BY product_id, valid_from_utc
            ORDER BY source_row_id DESC
        ) AS rn
    FROM silver.product_categories
),

cat_by_day AS (
    SELECT
        d.product_id,
        d.metric_date,
        c.category_name
    FROM product_dates d
    LEFT JOIN cats c
        ON c.product_id = d.product_id
        AND d.metric_date >= c.valid_from_utc::date
        AND (
            c.valid_to_utc IS NULL
            OR d.metric_date < c.valid_to_utc::date
        )
        AND c.rn = 1
),

promo AS (
    SELECT
        oi.product_id,
        o.order_date::date AS metric_date,
        COUNT(DISTINCT op.promotion_id) AS promotion_count
    FROM silver.order_items oi
    JOIN _tmp_order_360 o
        ON o.order_id = oi.order_id
    LEFT JOIN silver.order_promotions op
        ON op.order_id = oi.order_id
    GROUP BY
        oi.product_id,
        o.order_date::date
),

grid AS (
    SELECT
        product_id,
        metric_date
    FROM product_dates

    UNION

    SELECT
        product_id,
        metric_date
    FROM sales
),

sales_discount AS (
    SELECT
        oi.product_id,
        o.order_date::date AS metric_date,
        SUM(oi.item_discount_amount) AS discount_amount
    FROM silver.order_items oi
    JOIN _tmp_order_360 o
        ON o.order_id = oi.order_id
    GROUP BY
        oi.product_id,
        o.order_date::date
),

refund_amounts AS (
    SELECT
        oi.product_id,
        o.order_date::date AS metric_date,
        SUM(
            CASE
                WHEN o.refunded_amount > 0
                THEN oi.quantity * oi.unit_price
                ELSE 0
            END
        ) AS refund_amount
    FROM silver.order_items oi
    JOIN _tmp_order_360 o
        ON o.order_id = oi.order_id
    GROUP BY
        oi.product_id,
        o.order_date::date
)

SELECT
    g.product_id,
    g.metric_date,
    c.category_name AS active_category,

    COALESCE(s.units_sold, 0) AS units_sold,
    COALESCE(s.order_count, 0) AS order_count,
    COALESCE(s.gross_revenue, 0) AS gross_revenue,

    COALESCE(s.gross_revenue, 0)
        - COALESCE(sd.discount_amount, 0)
        - COALESCE(ra.refund_amount, 0) AS net_revenue,

    COALESCE(ru.refunded_units, 0) AS refunded_units,

    pd.available_inventory,
    pd.reserved_inventory,

    CASE
        WHEN pd.available_inventory IS NULL THEN NULL
        ELSE pd.available_inventory <= 0
    END AS stockout_flag,

    COALESCE(p.promotion_count, 0) AS promotion_count

FROM grid g

LEFT JOIN sales s
    ON s.product_id = g.product_id
    AND s.metric_date = g.metric_date

LEFT JOIN cat_by_day c
    ON c.product_id = g.product_id
    AND c.metric_date = g.metric_date

LEFT JOIN product_dates pd
    ON pd.product_id = g.product_id
    AND pd.metric_date = g.metric_date

LEFT JOIN refund_units ru
    ON ru.product_id = g.product_id
    AND ru.metric_date = g.metric_date

LEFT JOIN promo p
    ON p.product_id = g.product_id
    AND p.metric_date = g.metric_date

LEFT JOIN sales_discount sd
    ON sd.product_id = g.product_id
    AND sd.metric_date = g.metric_date

LEFT JOIN refund_amounts ra
    ON ra.product_id = g.product_id
    AND ra.metric_date = g.metric_date

ORDER BY
    g.product_id,
    g.metric_date

""", engine)

# Ensure numeric/integer contract columns
for c in [
    'units_sold',
    'order_count',
    'refunded_units',
    'promotion_count'
]:
    product_daily[c] = (
        product_daily[c]
        .fillna(0)
        .astype(int)
    )

print(
    "product_daily:",
    len(product_daily),
    "rows /",
    product_daily.product_id.nunique(),
    "products"
)

product_daily: 7201 rows / 81 products


## 2. Contract types + pipeline_run_id

In [5]:
for c in ["units_sold","order_count","refunded_units","promotion_count"]:
    product_daily[c] = product_daily[c].fillna(0).astype(int)
for c in ["available_inventory","reserved_inventory"]:
    product_daily[c] = product_daily[c].astype("Int64")
product_daily["pipeline_run_id"] = pipeline_run_id


## 3. Recreate target table sesuai `gold.sql`

In [6]:
gold_ddl = """CREATE SCHEMA IF NOT EXISTS gold;
DROP TABLE IF EXISTS gold.product_daily;
CREATE TABLE gold.product_daily (
    product_id TEXT NOT NULL,
    metric_date DATE NOT NULL,
    active_category TEXT,
    units_sold INTEGER NOT NULL,
    order_count INTEGER NOT NULL,
    gross_revenue NUMERIC(14,2) NOT NULL,
    net_revenue NUMERIC(14,2) NOT NULL,
    refunded_units INTEGER NOT NULL,
    available_inventory INTEGER,
    reserved_inventory INTEGER,
    stockout_flag BOOLEAN,
    promotion_count INTEGER NOT NULL,
    pipeline_run_id TEXT NOT NULL,
    PRIMARY KEY (product_id, metric_date)
);"""
with engine.begin() as conn:
    conn.execute(text(gold_ddl))

product_daily.to_sql("product_daily", engine, schema="gold", if_exists="append", index=False, method="multi")
print("gold.product_daily berhasil dibuat.")

gold.product_daily berhasil dibuat.


## 4. Final validation

In [7]:
df = pd.read_sql("SELECT * FROM gold.product_daily", engine)
print("Rows:", len(df))
print("Unique product-date:", df[["product_id","metric_date"]].drop_duplicates().shape[0])
print("Duplicate rows:", len(df) - len(df.drop_duplicates()))
print("NULL values:", int(df.isna().sum().sum()))


Rows: 7201
Unique product-date: 7201
Duplicate rows: 0
NULL values: 10
